# Notebook 1 — Exploring Relationships between Variables

**Dataset:** two years (731 days) of daily operations for a small café — weather, sales, customers, promotions and profit.
All data is synthetic, so every relationship you find was built in on purpose.

**What you will learn**
1. Why it is useful to look for relationships between variables
2. The difference between an *independent* and a *dependent* variable
3. How to read a scatter plot for **direction**, **strength** and **form**

## Setup
Run this cell first (and again after restarting the kernel). It imports the libraries we need and loads
`cafe_daily.csv`, which must sit in the same folder as this notebook.

- **pandas / numpy** – tables and numerical work
- **matplotlib / seaborn** – plotting
- **scipy.stats** – correlation and regression helpers

`cafe.head()` shows the first five rows so you can check the data loaded correctly.
See `data_dictionary.md` for what every column means.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid")

cafe = pd.read_csv("cafe_daily.csv", parse_dates=["date"])
cafe.head()

## Part 1 — Why Relationships between Variables Matter

A **relationship** between two variables means that knowing one tells you something about the other.
In a café, temperature might tell us how many iced drinks we will sell. Finding these links helps us
ask better questions, plan stock and staff, and later build predictive models.

But a relationship is only a starting point: it always needs interpretation in context
(Notebook 3 shows how easily it can mislead).

First, get a feel for the data. `describe()` gives summary statistics for every column:

| Statistic | Formula / meaning |
|---|---|
| Mean | $\bar{x} = \dfrac{1}{n}\sum_{i=1}^{n} x_i$ |
| Standard deviation | $s = \sqrt{\dfrac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2}$ — the typical distance from the mean |
| min / 25% / 50% / 75% / max | the spread of values; 50% is the median |

In [ ]:
cafe.describe(include='all').T

Below is a quick **correlation heatmap**. Each square shows how strongly two columns move together,
from −1 (dark blue, move in opposite directions) to +1 (dark red, move together). White means little or no
linear relationship. Don't worry about the exact numbers yet — Notebook 2 explains how they are calculated.

> **Things to notice:** which pairs are strongly red or blue? Which columns (e.g. `barista_shoe_size`) are pale everywhere?

In [ ]:
# A first, deliberately quick look — we unpack all of this properly later
num = cafe.select_dtypes('number').drop(columns=['is_weekend','promo_active'])
sns.heatmap(num.corr(), cmap='coolwarm', center=0, annot=False)
plt.title('Teaser: what moves with what?'); plt.show()

## Part 2 — Independent and Dependent Variables

| Term | Also called | Role | Café example |
|---|---|---|---|
| **Independent variable** ($x$) | input, predictor, explanatory variable | the thing we think *explains* the change | `temperature_c` |
| **Dependent variable** ($y$) | output, response, target | the outcome that *depends on* the input | `iced_drink_sales` |

We cannot control the weather, but we can observe it — and we suspect iced-drink sales depend on it.
By convention the independent variable goes on the **x-axis** and the dependent variable on the **y-axis**.

If the relationship is roughly a straight line we can describe it as $y \approx \beta_0 + \beta_1 x$,
where $\beta_1$ (the slope) is the change in $y$ for each one-unit increase in $x$. We fit this line in Notebook 4.

> **Note:** calling a variable "independent" is a modelling choice. It does not prove that $x$ *causes* $y$.

In [ ]:
# Independent (x) vs dependent (y)
sns.scatterplot(data=cafe, x='temperature_c', y='iced_drink_sales', alpha=.5)
plt.xlabel('Temperature (°C)  — independent')
plt.ylabel('Iced drink sales — dependent')
plt.title('Does the input (temperature) explain the outcome (iced drinks)?'); plt.show()

**Reading the plot:** the cloud of points rises from left to right — warmer days come with more iced-drink
sales. The points form a fairly tight band, so temperature explains a lot of the variation in iced-drink sales.
Look for the single point far above the band on a cold day; we come back to it below.

## Part 3 — Interpreting Scatter Plots

Always **look at the data before you calculate anything**. When reading a scatter plot, check three things:

| Feature | Question to ask | Possible answers |
|---|---|---|
| **Direction** | As $x$ increases, what happens to $y$? | positive (up), negative (down), none |
| **Strength** | How closely do points follow a pattern? | strong (tight band), weak (wide cloud), none |
| **Form** | What shape is the pattern? | linear (straight line), non-linear (curve), clusters |

Also look for **outliers** — points that sit far away from the overall pattern.

The four panels below show the four main cases:
- **Positive:** temperature vs iced drinks — both rise together
- **Negative:** temperature vs hot drinks — as one rises, the other falls
- **Near-zero:** barista shoe size vs revenue — a shapeless cloud, no relationship
- **Non-linear:** discount vs profit — profit rises then falls (an arc / inverted U)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 8))
sns.scatterplot(data=cafe, x='temperature_c', y='iced_drink_sales', alpha=.4, ax=ax[0,0])
ax[0,0].set_title('Positive — temp vs iced drinks')
sns.scatterplot(data=cafe, x='temperature_c', y='hot_drink_sales', alpha=.4, ax=ax[0,1])
ax[0,1].set_title('Negative — temp vs hot drinks')
sns.scatterplot(data=cafe, x='barista_shoe_size', y='daily_revenue', alpha=.4, ax=ax[1,0])
ax[1,0].set_title('Near-zero — shoe size vs revenue')
sns.scatterplot(data=cafe, x='discount_pct', y='daily_profit', alpha=.4, ax=ax[1,1])
ax[1,1].set_title('Non-linear — discount vs profit (note the arc)')
plt.tight_layout(); plt.show()

**What the panels tell us**
- The positive and negative panels both show strong, linear relationships — just in opposite directions.
- Shoe size vs revenue is a random cloud: knowing shoe size tells you nothing about revenue.
- Discount vs profit has a clear pattern, but it is **curved**. Small discounts raise profit, large ones
  reduce it. A straight-line measure will miss this (see Notebook 2, Part 3).

The next cell lists the three days with the highest iced-drink sales. One of them is clearly unusual.

In [ ]:
# The off-trend point in the iced-drinks plot is a real outlier (a one-off catering order).
# Notebook 2 measures how much it distorts the correlation.
cafe.nlargest(3, 'iced_drink_sales')[['date','temperature_c','iced_drink_sales','daily_revenue']]

**The outlier:** a cold February day (row 40) with a huge iced-drink order. It was a one-off catering order,
so it does not follow the temperature trend. Outliers like this can distort summary numbers — Notebook 2
shows exactly how much.

### Key takeaways
- Independent variable ($x$) = input/explanation; dependent variable ($y$) = outcome.
- Describe every scatter plot by **direction, strength and form**, and check for **outliers**.
- Plot first, calculate second.